# Capstone — From processed data to place cells

We bring together everything from the week:

- **Minian** → calcium traces + footprints (`C.zarr`, `A.zarr`, `max_proj.zarr`)
- **CaTune / CaDecon** → deconvolved neural activity
- **eztrack** → animal position
- **DAQ** → neural + behavior timestamps

and use **CaMAP** to fuse neural activity with behavior on a common clock, then
find and quantify place cells.

We run the pipeline *live*, pausing at each step to look at what it produced.

> Interactive widgets need Jupyter Lab: `jupyter lab` from the repo root.

## Setup & data paths

Pick the session. Inputs come from your own upstream runs or from Zenodo —
`python scripts/get_data.py` populates them local-first (see `data/README.md`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # make workshop_glue importable

SESSION = "prerecorded"        # or "live" for the workshop recording
SESS = Path(f"../../data/sessions/{SESSION}")

MINIAN_OUT = SESS / "minian_out"
DECONV_OUT = SESS / "deconv_out"        # calab: CaTune / CaDecon
EZTRACK_CSV = SESS / "eztrack_out" / "position.csv"
NEURAL_TS = SESS / "raw" / "neural_timestamp.csv"
BEHAVIOR_TS = SESS / "raw" / "behavior_timestamp.csv"

CONFIG = Path("config/arena_config.yaml")      # TODO: create for the example session
DATA_PATHS = Path("config/data_paths.yaml")    # TODO: create for the example session

# Sanity check: if a stage is missing, run `python scripts/get_data.py`.
for p in (MINIAN_OUT, DECONV_OUT, EZTRACK_CSV, NEURAL_TS, BEHAVIOR_TS):
    if not p.exists():
        print(f"MISSING {p}  ->  run: python scripts/get_data.py --session {SESSION}")

## 1. Glue: eztrack → DeepLabCut format

eztrack writes a flat `frame, x, y` CSV; CaMAP expects a DeepLabCut-style
3-header CSV. A good moment to talk about position-format conventions.

In [ ]:
from workshop_glue import eztrack_to_dlc

dlc_csv = eztrack_to_dlc(
    EZTRACK_CSV,
    DATA / 'eztrack_out' / 'position_dlc.csv',
    bodypart='LED',  # must match behavior.bodypart in the data config
)
print('Wrote', dlc_csv)

## 2. `load()` — what Minian gave us

Build the dataset from config and load traces, footprints, and the max
projection. Inspect a few calcium traces.

In [ ]:
import matplotlib.pyplot as plt
from camap.dataset import BaseCaMAPDataset

ds = BaseCaMAPDataset.from_yaml(CONFIG, DATA_PATHS)
ds.load()
print(f'{ds.traces.sizes["unit_id"]} units x {ds.traces.sizes["frame"]} frames')

for uid in ds.traces['unit_id'].values[:5]:
    plt.plot(ds.traces.sel(unit_id=uid).values, lw=0.6)
plt.title('Example Minian calcium traces'); plt.xlabel('frame'); plt.show()

## 3. `preprocess_behavior()` — clean the trajectory

Hampel jump removal, perspective correction, arena clipping, mm conversion.
`plot_preprocess_steps` shows the trajectory at each stage.

In [ ]:
ds.preprocess_behavior()

from camap.visualization import plot_preprocess_steps
plot_preprocess_steps(ds); plt.show()  # TODO: confirm signature against pinned CaMAP

## 4. Inject deconvolved neural activity (calab: CaTune / CaDecon)

Deconvolution is an explicit upstream stage — Minian's own deconvolution is
skipped and calab (CaTune or CaDecon) produced the deconvolved estimate of
neural activity, saved as `deconv_out/activity.npy`. So instead of CaMAP's
built-in OASIS (`ds.deconvolve()`), the glue loads that file and sets
`ds.good_unit_ids` / `ds.S_list` directly (rows aligned to `ds.traces`).

> Produce `activity.npy` with `tutorials/deconvolution/run_cadecon.py` (or
> `run_catune.py`), or fetch the `deconv_out` bundle via `scripts/get_data.py`.
> Fallback: `ds.deconvolve(progress_bar=tqdm)` with OASIS params in the config.

In [ ]:
from workshop_glue import inject_deconv_activity

inject_deconv_activity(ds, DECONV_OUT)
print(f'Injected deconvolved activity for {len(ds.good_unit_ids)} units')

# Fallback (CaMAP's own OASIS):
# from tqdm.auto import tqdm
# ds.deconvolve(progress_bar=tqdm)

## 5. `match_events()` — fuse neural + behavior on one clock

The conceptual heart: behavior is interpolated onto neural timestamps to build
the **canonical neural-rate table** (`ds.canonical`) — one row per neural frame.

In [ ]:
ds.match_events()
ds.canonical.head()

## 6. `compute_occupancy()` — where the animal spent time

In [ ]:
ds.compute_occupancy()

from camap.visualization import plot_occupancy_preview
plot_occupancy_preview(ds); plt.show()  # TODO: confirm signature

## 7. Metrics deep-dive (one example unit)

Open the hood on a single unit: rate map → Skaggs spatial information →
circular-shift shuffle null → p-value. Uses the individually-callable functions
in `camap.analysis.spatial_2d` so the math is visible, not hidden in
`analyze_units()`.

In [ ]:
# TODO: pick an example unit and walk through:
#   compute_rate_map -> compute_spatial_information -> compute_stability_score
# plotting the rate map and the shuffle distribution with the observed SI marked.
from camap.analysis import spatial_2d  # noqa: F401

## 8. `analyze_units()` — all units + summary

Rate maps, spatial information, stability splits, and place-field detection for
every unit, then the population summary.

In [ ]:
from tqdm.auto import tqdm
ds.analyze_units(progress_bar=tqdm)

s = ds.summary()
print(f"{s['n_total']} units | {s['n_sig']} sig | {s['n_stable']} stable | "
      f"{s['n_place_cells']} place cells ({s['pct_place_cells']}%)")

from camap.visualization import plot_summary_scatter, plot_diagnostics
plot_diagnostics(ds.unit_results, p_value_threshold=ds.p_value_threshold); plt.show()

## 9. Explore — interactive unit browser

In [ ]:
%matplotlib widget
from camap.notebook import browse_units

fig, controls = browse_units(ds)
from IPython.display import display
display(controls)

## 10. `save_bundle()` — persist results

Writes a self-contained `.camap` directory you can reopen later in CaMAP's
`view_results_arena.ipynb` viewer.

In [ ]:
bundle = ds.save_bundle('output/workshop_demo')
print('Saved', bundle)